# Student Earnings Pipeline

### Setup

Download the `student-data.zip` folder from the google drive and extract it in the project directory.
The directory structure for this pipline should look like the following (place your extracted files into the raw folder):

```md
data/
    raw/
        Most-Recent-Cohorts-Field-of-Study.csv
        Most-Recent-Cohorts-Institution.csv
        CW2024_prelim.xlsx
        earning_buckets.csv
    processed/
data-pipline
    notebooks.ipynb
```
Install the required packages listed in `requirements.txt`

Outputs are saved to `data/processed/`

In [1]:
import polars as pl
import pandas as pd
from pathlib import Path

# Use the current working directory as the starting point for data paths
BASE_DIR = Path.cwd()

RAW_DATA_DIR = BASE_DIR / ".." / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / ".." / "data" / "processed"


# Create the input/ouput folders if they do not already exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


## Initial Raw Data Processing

In [2]:
def main():
    print("Scanning CSV files...")

    horizons = ["1YR", "4YR", "5YR"]

    # Cols holding "PS" (privacy suppressed), must be read as String first
    suppressed_cols = (
        [f"EARN_MDN_{h}" for h in horizons]
        + [f"EARN_COUNT_WNE_{h}" for h in horizons]
        + ["IPEDSCOUNT1"]
    )

    # Scorecard writes literal "NA" in some cols, including UNITID
    nulls = ["NA", "NULL", ""]

    # Helper to turn suppressed text into numbers
    def clean_suppressed(col_name, dtype=pl.Float64):
        return (
            pl.col(col_name)
            .str.strip_chars()
            .replace(["PS", ""], None)
            .cast(dtype, strict=False)
        )

    # Showing better documented CA public universities for demo purposes
    systems = {
        "California State University": "CSU",
        "University of California": "UC",
    }
    ca = (
        pl.read_excel(RAW_DATA_DIR / "CW2024_prelim.xlsx", sheet_name="Crosswalk")
        .select(
            pl.col("IPEDSMatch").cast(pl.Int64, strict=False).alias("UNITID"),
            pl.col("f1sysnam").replace_strict(systems, default=None).alias("system"),
        )
        .drop_nulls()
        .unique(subset="UNITID")
    )

    # Institution metadata
    meta = pl.scan_csv(
        RAW_DATA_DIR / "Most-Recent-Cohorts-Institution.csv",
        schema_overrides={c: pl.String for c in ["UGDS"]},
        null_values=nulls,
        infer_schema_length=10000,
    ).select(
        pl.col("UNITID").cast(pl.Int64),
        pl.col("CITY").alias("city"),
        pl.col("STABBR").alias("state"),
        pl.col("UGDS").cast(pl.Int32, strict=False).alias("undergrad_enrollment"),
    )

    df = pl.scan_csv(
        RAW_DATA_DIR / "Most-Recent-Cohorts-Field-of-Study.csv",
        schema_overrides={c: pl.String for c in suppressed_cols + ["CIPCODE"]},
        null_values=nulls,
        infer_schema_length=10000,
    )

    df = (
        df.with_columns(pl.col("UNITID").cast(pl.Int64))
        .join(ca.lazy(), on="UNITID", how="inner")
        .join(meta, on="UNITID", how="left")
        .with_columns(
            # Keep as text, CIP codes can start with a zero (0110)
            pl.col("CIPCODE").str.zfill(4).alias("cipcode"),

            *[clean_suppressed(f"EARN_MDN_{h}").alias(f"earn_mdn_{h.lower()}")
              for h in horizons],
            *[clean_suppressed(f"EARN_COUNT_WNE_{h}", pl.Int32).alias(f"earn_count_{h.lower()}")
              for h in horizons],
            clean_suppressed("IPEDSCOUNT1", pl.Int32).alias("completions"),

            # Why a value is missing: "PS" = withheld for privacy, blank = not reported
            *[pl.when(pl.col(f"EARN_MDN_{h}") == "PS").then(pl.lit("suppressed"))
               .when(pl.col(f"EARN_MDN_{h}").is_null()).then(pl.lit("not_reported"))
               .otherwise(pl.lit("usable"))
               .cast(pl.Categorical).alias(f"status_{h.lower()}")
              for h in horizons],
        )
        .select(
            pl.col("UNITID").alias("unitid"),
            pl.col("INSTNM").alias("institution"),
            "system", "city", "state", "undergrad_enrollment",
            "cipcode",
            pl.col("CIPDESC").alias("program"),
            pl.col("CREDLEV").cast(pl.Int16).alias("credlev"),
            "completions",
            *[f"earn_mdn_{h.lower()}" for h in horizons],
            *[f"earn_count_{h.lower()}" for h in horizons],
            *[f"status_{h.lower()}" for h in horizons],
        )
        .sort(["system", "institution", "cipcode", "credlev"])
    )

    final_df = df.collect()
    final_df.write_parquet(PROCESSED_DATA_DIR / "student-earnings-intermediate.parquet", compression="zstd")

    print(f"Processed {final_df.height:,} records.")
    schema = final_df.schema
    for col_name, dtype in schema.items():
        print(f"{col_name}: {dtype}")


if __name__ == "__main__":
    main()

Could not determine dtype for column 14, falling back to string


Scanning CSV files...
Processed 4,504 records.
unitid: Int64
institution: String
system: String
city: String
state: String
undergrad_enrollment: Int32
cipcode: String
program: String
credlev: Int16
completions: Int32
earn_mdn_1yr: Float64
earn_mdn_4yr: Float64
earn_mdn_5yr: Float64
earn_count_1yr: Int32
earn_count_4yr: Int32
earn_count_5yr: Int32
status_1yr: Categorical
status_4yr: Categorical
status_5yr: Categorical


### Assign Departments

In [3]:
earnings = pd.read_parquet(PROCESSED_DATA_DIR / "student-earnings-intermediate.parquet")
buckets = pd.read_csv(RAW_DATA_DIR / "earning_buckets.csv")

# Clean accidental spaces so program names match reliably
earnings["program_key"] = earnings["program"].str.strip()
buckets["program_key"] = buckets["program"].str.strip()

# Keep only the lookup columns, then add department to every earnings row
earnings = earnings.drop(columns="department", errors="ignore")

earnings_with_buckets = earnings.merge(
    buckets[["program_key", "department"]],
    on="program_key",
    how="left",       # keep every row from earnings
    validate="m:1"    # many earnings rows can map to one program bucket
)

earnings_with_buckets = earnings_with_buckets.drop(columns="program_key")

earnings_with_buckets.to_parquet(
    PROCESSED_DATA_DIR / "golden-student-earnings-with-buckets.parquet",
    index=False
)

print("Unmatched rows:", earnings_with_buckets["department"].isna().sum())

Unmatched rows: 8


### Prepare the Analysis Dataset

- remove columns that are not required for the earnings summaries

In [4]:
needs_cleaning = pd.read_parquet(PROCESSED_DATA_DIR / "golden-student-earnings-with-buckets.parquet")

needs_cleaning = needs_cleaning.drop(columns=[
    "unitid",
    "system",
    "city",
    "state",
    "undergrad_enrollment",
    "cipcode",
    "credlev",
    "completions",
    "status_1yr",
    "status_4yr",
    "status_5yr"
])

### Calculate Earnings Statistics

- shows the mean, median, standard deviation, and 25th and 75th percentiles of program level median earnings by institution and department
- missing earnings are excluded

In [5]:
count_cols = [
    "earn_mdn_1yr",
    "earn_mdn_4yr",
    "earn_mdn_5yr"
]

df = needs_cleaning.copy()

# Convert values to numeric. Keep missing values as NaN:
# Pandas excludes NaN values from mean, median, std, and quantiles.
for col in count_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

grouped = df.groupby(["institution", "department"])


stats_parts = []

for col in count_cols:
    stats = grouped[col].agg(
        mean="mean",
        median="median",
        std="std",
        quantile_25=lambda x: x.quantile(0.25),
        quantile_75=lambda x: x.quantile(0.75),
    )

    # Makes names such as mean_earn_count_1yr
    stats.columns = [f"{stat}_{col}" for stat in stats.columns]
    stats_parts.append(stats)

student_stats = (
    pd.concat(stats_parts, axis=1)
    .reset_index()
    .rename(columns={"institution": "Institution", "department": "department"})
)

student_stats.to_parquet(
    PROCESSED_DATA_DIR / "golden-student-pay-stats.parquet",
    index=False
)

display(student_stats)

,Institution,department,mean_earn_mdn_1yr,median_earn_mdn_1yr,std_earn_mdn_1yr,quantile_25_earn_mdn_1yr,quantile_75_earn_mdn_1yr,mean_earn_mdn_4yr,median_earn_mdn_4yr,std_earn_mdn_4yr,quantile_25_earn_mdn_4yr,quantile_75_earn_mdn_4yr,mean_earn_mdn_5yr,median_earn_mdn_5yr,std_earn_mdn_5yr,quantile_25_earn_mdn_5yr,quantile_75_earn_mdn_5yr
0,Cal Poly Maritime Academy,"Agriculture, Food & Animal",101735.000000,101735.0,NaN,101735.00,101735.00,117176.000000,117176.0,NaN,117176.00,117176.00,122033.0,122033.0,NaN,122033.00,122033.00
1,Cal Poly Maritime Academy,"Business, Economics & Finance",79890.000000,79890.0,NaN,79890.00,79890.00,93102.500000,93102.5,24789.042428,84338.25,101866.75,108546.5,108546.5,41942.038726,93717.75,123375.25
2,Cal Poly Maritime Academy,Engineering,97760.500000,97760.5,796.909342,97478.75,98042.25,126302.000000,126302.0,131.521861,126255.50,126348.50,128170.0,128170.0,4802.669258,126472.00,129868.00
3,Cal Poly Maritime Academy,Humanities & Languages,39705.000000,39705.0,NaN,39705.00,39705.00,76478.000000,76478.0,NaN,76478.00,76478.00,90500.0,90500.0,NaN,90500.00,90500.00
4,Cal Poly Maritime Academy,Science & Mathematics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,University of California-Santa Cruz,Engineering,72918.000000,72918.0,NaN,72918.00,72918.00,98567.000000,98567.0,NaN,98567.00,98567.00,119939.0,119939.0,NaN,119939.00,119939.00
321,University of California-Santa Cruz,Humanities & Languages,33869.818182,33889.0,6540.358871,29223.00,39532.00,52737.666667,53658.5,10416.214653,48776.00,58752.00,56744.2,56152.5,8134.869524,52469.75,62385.75
322,University of California-Santa Cruz,Medicine & Heath Sciences,45764.000000,45764.0,NaN,45764.00,45764.00,69027.000000,69027.0,NaN,69027.00,69027.00,69052.0,69052.0,NaN,69052.00,69052.00
323,University of California-Santa Cruz,Science & Mathematics,52350.555556,52133.0,14431.969686,41374.00,56661.00,67305.625000,67442.5,11067.371219,63177.75,70616.75,82706.0,78785.5,21800.505826,75471.50,84309.00


In [6]:
salaries = pd.read_parquet(PROCESSED_DATA_DIR / "golden-student-earnings-with-buckets.parquet")

salaries.columns

Index(['unitid', 'institution', 'system', 'city', 'state',
       'undergrad_enrollment', 'cipcode', 'program', 'credlev', 'completions',
       'earn_mdn_1yr', 'earn_mdn_4yr', 'earn_mdn_5yr', 'earn_count_1yr',
       'earn_count_4yr', 'earn_count_5yr', 'status_1yr', 'status_4yr',
       'status_5yr', 'department'],
      dtype='str')